In [1]:
import json
from tqdm import tqdm_notebook as tqdm
from collections import defaultdict
# from tqdm import tqdm as tqdm

In [2]:
import pandas as pd
from functools import reduce
from collections import Counter
import numpy as np

In [3]:
import os

In [4]:
def handle_attributes(ann,
    single_choice_attributes = ['department','closure','type','style'],
    multiple_choice_attributes = []
     ):
    '''
    this takes care of what attributes are supposed to be single-choice,
    which ones are multiple-choice etc
    '''
    for attr_name in ann:
        if attr_name in single_choice_attributes:
            ann[attr_name] = ann[attr_name][0]
            pass
        elif attr_name in multiple_choice_attributes:
            pass
    return ann

In [15]:
'''
This method parses json annotation file
and return key value pairs
'''
def parse_single_json(fname,handle_attributes=lambda ann:ann):
    with open(fname,'r') as f:
        J = json.load(f)
#     import pdb;pdb.set_trace()
    was_cancelled = J['was_cancelled']
    is_labelled = J['task']['is_labeled']
    data = J['task']['data']['data']
    out = {}
    out['data'] = data
    if not was_cancelled and is_labelled:
        ann = J['result']
        for item in ann:
            attribute_name = item['from_name']
            attribute_name = attribute_name.replace('hoodie-','').replace('jacket-','')
            value = item['value']
            if 'choices' in value:
                attribute_value = value['choices']
            out[attribute_name] = attribute_value
        pass
    
    out = handle_attributes(out)
    return out

In [36]:
'''
This method reads through annotation folder
and parses all annotations done by annotators
and returns list of dataframe containing parsed iformation
'''

def read_all_annotators_data(
    annotation_results_folder,
    prefix_to_exclude_demo = 'annotator_'):

    annotator_folders = os.listdir(annotation_results_folder)
    annotator_folders = list(sorted(annotator_folders))
    all_annotations = []
    for person_i,annotator_folder in enumerate(tqdm(annotator_folders)):

        if not annotator_folder.startswith(prefix_to_exclude_demo):
            continue
        full_annotator_folder = os.path.join(annotation_results_folder,annotator_folder)
        fname_annotated_jsons = [fname for fname in os.listdir(full_annotator_folder) if fname.endswith('.json')]
        full_fname_annotated_jsons = [os.path.join(full_annotator_folder,fname) for fname in fname_annotated_jsons]
        annotated_by_this_annotator = []
        for jname in full_fname_annotated_jsons:
            ann = parse_single_json(jname,handle_attributes=handle_attributes)
            ann['json-fname'] = jname
            annotated_by_this_annotator.append(ann)
        all_annotations.append(annotated_by_this_annotator)
        '''
        if person_i == 5:
            # for debugging
            break
        '''
    return all_annotations

In [37]:
'''
If there is repetition on annotations then we can keep most recent annotation
Here what does repition mean ?
'''

def keep_only_most_recent_annotation(annotations):
    imnames = [ann_i['data'] for ann_i in annotations]
#     import pdb;pdb.set_trace()
    indexes = range(len(imnames))
    imname_vs_index = defaultdict(list)
    for n,i in zip(imnames,indexes):
        imname_vs_index[n].append(i)
    indexes_to_keep = []
    for (imname,locs) in (imname_vs_index.items()):
        if len(locs) == 1:
            indexes_to_keep.append(locs[0])
        else:
            '''
            example time string: 2021-11-08T04:41:37.086182Z
            '''
            year = "0000"
            month = "11"
            day = "08"
            hour = "00"
            minute = "00"
            latest_loc = -1
            for li in locs:
                json_fname = annotations[li]['json-fname']
                
                with open(json_fname,'r') as f:
                    J = json.load(f)
                    J = J['updated_at']
                    J_year = J[:len(year)]
                    J_month = J[len(year + '-'):len(year + '-')+len(month)]
                    J_day = J[len(year + '-')+len(month + '-'):len(year + '-')+len(month + '-') + len(day)]
                    J_hour = J[len(year + '-')+len(month + '-') + len(day + 'T'):len(year + '-')+len(month + '-') + len(day + 'T') + len(hour)]
                    J_minute = J[len(year + '-')+len(month + '-') + len(day + 'T') + len(hour+":"):len(year + '-')+len(month + '-') + len(day + 'T') + len(hour+":")+len(minute)]
                    #..............................................
                    if int(year) > int(J_year):
#                         print(f'continuing from year {year},{J_year}')
                        continue
                    if (int(year) == int(J_year)):
                        if (int(month)> int(J_month)):
#                             print(f'continuing from month {month},{J_month}')
                            continue                          
                        if (int(month) == int(J_month)):
                            if (int(day) > int(J_day)):
#                                 print(f'continuing from day {day},{J_day}')
                                continue
                            if (int(day) == int(J_day)):
                                if (int(hour) > int(J_hour)):
#                                     print(f'continuing from hour {hour},{J_hour}')
                                    continue
                                if (int(hour) == int(J_hour)):
                                    if (int(minute) > int(J_minute)):
#                                         print(f'continuing from minute {minute},{J_minute}')
                                        continue
                    #..............................................
                    year = J_year
                    month = J_month
                    day = J_day
                    hour = J_hour
                    minute = J_minute
                    latest_loc = li
                    #..............................................
            assert latest_loc > -1
            indexes_to_keep.append(latest_loc)
#     indexes_to_keep = list(sorted(indexes_to_keep))
    '''
    indexes_to_drop = list(set(indexes).difference(indexes_to_keep))
    df_kep = df_annotations[df_annotations.iloc is in ]
    '''
    kept = [annotations[i]  for i in indexes_to_keep]
    return kept

In [6]:
def outer_merge_annotations(annotations):  
    # will take per-annotator dataframes
    # and create a new data frame where
    # the closure choice of annotator 0 is closure_0
    # the closure choice of annotator 1 is closure_1
    # etc, and similarly for all other attributes
    
    df_merged = annotations[0].copy()
    for i,df_next in enumerate(annotations[1:]):
        df_merged = df_merged.merge(df_next,on='data',suffixes=(f'',f'_{i+1}'),how='outer')

    return df_merged
    pass

In [84]:
'''
What is purpose of this aggregation function
'''

'''
it takes all_annotations as input (here annotatons mean list of list of annotations from 50 annotators)
Fileds

'''

def aggregate_annotations(
    all_annotations,
    fields,
):
    '''
    to collect all annotators data, and 
    aggregate multiple annotations for 1 image
    '''
    
    all_annotations_non_empty = [ann for ann in all_annotations if len(ann) > 0]
    df_annotations = [pd.DataFrame.from_dict(ann) for ann in all_annotations_non_empty]
    #--------------------------------
    from collections import Counter,defaultdict
    
    #--------------------------------
    '''
    stop if any annotator has annotated the same image more than once
    this should have been addressed in an earlier step
    '''
    for dfi in df_annotations:   
        imnames = list(dfi.data)
        imname_locs = defaultdict(list)
        for i,n in enumerate(imnames):
            imname_locs[n].append(i)
        for n,locs in imname_locs.items():
            if len(locs) > 1:
                continue
                #import pdb;pdb.set_trace()
    #--------------------------------
    df_annotations = outer_merge_annotations(df_annotations)
    print('annotations merged')
    print(f'#annotations {len(df_annotations)}')


    for k in tqdm(fields):
        # sometimes labelstudio inexplicably puts the selected option
        # inside a list example: instead of 'closure' being 'zip'
        # it'll be ['zip'].
        # this takes care of that

        #---------------------------------
        def strip_list(el):
            if isinstance(el,list):
                assert len(el) == 1
                return el[0] 
            else: 
                return el
        #---------------------------------
        for column in df_annotations.columns: 
            if column.startswith(k):


                df_annotations[column] = df_annotations[column].apply(
                    strip_list,
                    )



        # example:combine all closure columns into 1 column
        df_annotations[k] = df_annotations.apply(lambda row:[row[column] for column in row.keys() if column.startswith(k) and not pd.isna(row[column])] ,
                                                     axis=1)
        print(f'{k} columns combined')
        #example: count unique closure values in each row
        df_annotations[k] = df_annotations.apply(lambda row:Counter(row[k]),
                                                     axis=1)
        print(f'{k} values counted')
        # example: pick the closure value with highest count, given it is larger than 2
        # assumption: if lace-up/tie has count of 2 , all other closures will have counts <= 1
        df_annotations[k] = df_annotations.apply(lambda row: max(row[k],key=row[k].get) \
                                                 if  ( len(row[k]) > 0 and (max(row[k].values()) > 1) ) else float('nan'),\
                                                  axis=1)
        print(f'most common {k} value chosen')
        print('-'*40)

    df_annotations_columns = df_annotations.columns
    for column in df_annotations_columns:
        for k in fields:
            if column.startswith(f'{k}_'):
                print(column,end=' ')
                df_annotations = df_annotations.drop([column],axis=1)
    imnames = list(df_annotations.data)
    uq_imnames = set(imnames)
#     if len(uq_imnames) != len(imnames):
#         import pdb;pdb.set_trace()
    return df_annotations

In [85]:
def remove_tag_choices(df_annotations):
    '''
    will remove the tag-choices column from a data frame
    '''
    for column in df_annotations.columns:
        for k in ['tag-choices']:
            if column.startswith(f'{k}_'):
                print(column,end=' ')
                df_annotations = df_annotations.drop([column],axis=1)
    return df_annotations


In [86]:
fields = ['department','length','sleeve-length','pattern','style']

In [87]:
all_annotations = read_all_annotators_data('./sample_annotations/')

<ipython-input-36-fa142d21f876>:14: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for person_i,annotator_folder in enumerate(tqdm(annotator_folders)):


In [88]:
aggregate_annotations(all_annotations,fields)

annotations merged
#annotations 5646


<ipython-input-84-4697b3d98a88>:45: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for k in tqdm(fields):


department columns combined
department values counted
most common department value chosen
----------------------------------------
length columns combined
length values counted
most common length value chosen
----------------------------------------
sleeve-length columns combined
sleeve-length values counted
most common sleeve-length value chosen
----------------------------------------
pattern columns combined
pattern values counted
most common pattern value chosen
----------------------------------------
style columns combined
style values counted
most common style value chosen
----------------------------------------

department_1 length_1 sleeve-length_1 pattern_1 style_1 

,data,department,length,sleeve-length,pattern,style,json-fname,json-fname_1
0,/data/local-files/?d=label-studio/data/Images/...,hoodie,Regular/Mid-length,Long sleeve,Graphic Print,Not-Applicable/Visible,./sample_annotations/annotator_0/71970.json,./sample_annotations/annotator_1/49504.json
1,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/122487.json,NaN
2,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/44957.json,NaN
3,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/66825.json,NaN
4,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/77091.json,NaN
...,...,...,...,...,...,...,...,...
5641,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_1/69130.json
5642,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_1/47174.json
5643,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_1/65304.json
5644,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_1/61165.json


In [20]:
parse_single_json('./sample_annotations/annotator_0/100209.json')

{'data': '/data/local-files/?d=label-studio/data/Images/random_jackets_350_Occasion--business_men_jackets--jackets_Occasion--business_men_jackets--local--Images--403211070949--0.jpg',
 'department': ['jacket'],
 'length': ['Regular/Mid-length'],
 'sleeve-length': ['Long sleeve'],
 'pattern': ['Solid'],
 'style': ['Not-Applicable/Visible']}

In [51]:
'''
parse all jsons availabel inside project
'''

path = './sample_annotations/annotator_0/'

jsonfiles = []
for dirpath, subdirs, files in os.walk(path):
    for x in files:
        if x.endswith(".json"):
            jsonfiles.append(os.path.join(dirpath, x))


In [52]:
annotations_0 = [parse_single_json(x) for x in jsonfiles]

In [53]:
annotations_0[0]

{'data': '/data/local-files/?d=label-studio/data/Images/random_jackets_350_Features--preshrunk_men_jackets--jackets_Features--preshrunk_men_jackets--local--Images--324457692801--0.jpg',
 'department': ['jacket'],
 'length': ['Regular/Mid-length'],
 'sleeve-length': ['Long sleeve'],
 'pattern': ['Solid'],
 'style': ['Not-Applicable/Visible']}

In [30]:
counter = 0
other_tag_images = []
for x in annotations:
    if 'other' in x['department'] or 'tag' in x['department']:
        counter+=1
        other_tag_images.append(x['data'].split('/')[-1])
print(counter)

4


In [31]:
other_tag_images

['random_jackets_350_Neckline--round_neck_men_jackets--jackets_Neckline--round_neck_men_jackets--local--Images--324763667923--5.jpg',
 'random_hoodies_350_Outer-Shell-Material--stone_men_hoodies--hoodies_Outer-Shell-Material--stone_men_hoodies--local--Images--363298709782--1.jpg',
 'TL2K4DSDfwCk-qy.jpg',
 'random_jackets_350_Length--long_men_jackets--jackets_Length--long_men_jackets--local--Images--234248855093--6.jpg']

In [77]:
aggregate_annotations(all_annotations,fields=['department','length','sleeve-length','pattern','style'])

annotations merged
#annotations 43


<ipython-input-7-2d66a119f06f>:34: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for k in tqdm(fields):


department columns combined
department values counted
most common department value chosen
----------------------------------------
length columns combined
length values counted
most common length value chosen
----------------------------------------
sleeve-length columns combined
sleeve-length values counted
most common sleeve-length value chosen
----------------------------------------
pattern columns combined
pattern values counted
most common pattern value chosen
----------------------------------------
style columns combined
style values counted
most common style value chosen
----------------------------------------



,data,department,length,sleeve-length,pattern,style,json-fname
0,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100705.json
1,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100318.json
2,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100585.json
3,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100572.json
4,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100132.json
5,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100785.json
6,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100274.json
7,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100295.json
8,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100011.json
9,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100187.json


In [48]:
all_annotations=read_all_annotators_data('./sample_annotations/')

<ipython-input-36-fa142d21f876>:14: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for person_i,annotator_folder in enumerate(tqdm(annotator_folders)):


In [50]:
aggregate_annotations(all_annotations=all_annotations,fields={'department','length','sleeve-length','pattern','style'})

annotations merged
#annotations 43


<ipython-input-7-2d66a119f06f>:34: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for k in tqdm(fields):


pattern columns combined
pattern values counted
most common pattern value chosen
----------------------------------------
style columns combined
style values counted
most common style value chosen
----------------------------------------
sleeve-length columns combined
sleeve-length values counted
most common sleeve-length value chosen
----------------------------------------
department columns combined
department values counted
most common department value chosen
----------------------------------------
length columns combined
length values counted
most common length value chosen
----------------------------------------



,data,department,length,sleeve-length,pattern,style,json-fname
0,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100705.json
1,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100318.json
2,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100585.json
3,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100572.json
4,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100132.json
5,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100785.json
6,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100274.json
7,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100295.json
8,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100011.json
9,/data/local-files/?d=label-studio/data/Images/...,NaN,NaN,NaN,NaN,NaN,./sample_annotations/annotator_0/100187.json


In [40]:
type(annotations)

list

dict

In [67]:
fields = ['department','length','sleeve-length','pattern','style']

In [68]:
for x in tqdm(fields):
    print(x)

<ipython-input-68-55c4976bbfc8>:1: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for x in tqdm(fields):


department
length
sleeve-length
pattern
style



In [74]:
all_annotations_non_empty = [ann for ann in all_annotations if len(ann) > 0]
df_annotations = [pd.DataFrame.from_dict(ann) for ann in all_annotations_non_empty] # one dataframe per annotator_ project

In [71]:
len(df_annotations)

1

In [63]:
from collections import Counter,defaultdict

In [72]:
for k in tqdm(fields):
        # sometimes labelstudio inexplicably puts the selected option
        # inside a list example: instead of 'closure' being 'zip'
        # it'll be ['zip'].
        # this takes care of that

        #---------------------------------
        def strip_list(el):
            if isinstance(el,list):
                assert len(el) == 1
                return el[0] 
            else: 
                return el
        #---------------------------------
        for column in df_annotations[0].columns: 
            if column.startswith(k):


                df_annotations[column] = df_annotations[column].apply(
                    strip_list,
                    )



        # example:combine all closure columns into 1 column
        df_annotations[k] = df_annotations.apply(lambda row:[row[column] for column in row.keys() if column.startswith(k) and not pd.isna(row[column])] ,
                                                     axis=1)
        print(f'{k} columns combined')
        #example: count unique closure values in each row
        df_annotations[k] = df_annotations.apply(lambda row:Counter(row[k]),
                                                     axis=1)
        print(f'{k} values counted')
        # example: pick the closure value with highest count, given it is larger than 2
        # assumption: if lace-up/tie has count of 2 , all other closures will have counts <= 1
        df_annotations[k] = df_annotations.apply(lambda row: max(row[k],key=row[k].get) \
                                                 if  ( len(row[k]) > 0 and (max(row[k].values()) > 1) ) else float('nan'),\
                                                  axis=1)
        print(f'most common {k} value chosen')
        print('-'*40)

<ipython-input-72-510403aff191>:1: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for k in tqdm(fields):


TypeError: list indices must be integers or slices, not str